# NYC Restaurant Health Inspections — Data Cleaning Capstone

**Skills applied:** handling missing values, scaling/parsing inconsistent text, parsing dates,
fixing data types, removing duplicates, detecting and handling outliers.

**Scenario:** A city health department's restaurant inspection export is messy — inconsistent
borough/cuisine spelling and casing, four different date formats, impossible inspection scores,
duplicate rows, and missing values. This notebook turns the raw export into an analysis-ready
dataset and documents every decision.

**Dataset:** Synthetically generated to mimic the structure and messiness of the real
[NYC Restaurant Inspection Results](https://www.kaggle.com/datasets/new-york-city/nyc-restaurant-inspection-results)
Kaggle dataset, so the notebook can run fully offline and reproducibly.


## 1. Load the raw data and get a first look

In [1]:
import pandas as pd
import numpy as np

raw = pd.read_csv("data/raw_restaurant_inspections.csv")
print(raw.shape)
raw.head(10)


(625, 8)


,inspection_id,restaurant_name,borough,cuisine_type,inspection_date,score,grade,violations_count
0,1000,Restaurant 32,BRONX,mexican,2023/10/20,51,NaN,6
1,1001,Restaurant 130,BRONX,Amer.,11/23/2022,32,A,7
2,1002,Restaurant 156,STATEN ISLAND,Jap.,25-09-2023,11,A,10
3,1003,Restaurant 130,Bronx,Mexican,23-02-2022,41,A,8
4,1004,Restaurant 294,Brooklyn,American,22-10-2024,40,A,6
5,1005,Restaurant 161,Manhattan,Chinese,18-07-2023,26,B,6
6,1006,Restaurant 122,Queens,Chinese,04/25/2024,24,A,6
7,1007,Restaurant 274,queens,Amer.,13-09-2021,19,B,1
8,1008,Restaurant 173,Queens,American,2022/02/10,18,A,5
9,1009,Restaurant 93,Bklyn,CHINESE,13-02-2022,47,C,2


In [2]:
raw.info()


<class 'pandas.DataFrame'>
RangeIndex: 625 entries, 0 to 624
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   inspection_id     625 non-null    int64
 1   restaurant_name   618 non-null    str  
 2   borough           625 non-null    str  
 3   cuisine_type      625 non-null    str  
 4   inspection_date   625 non-null    str  
 5   score             625 non-null    int64
 6   grade             596 non-null    str  
 7   violations_count  567 non-null    str  
dtypes: int64(2), str(6)
memory usage: 39.2 KB


## 2. Scope the damage

Before fixing anything, quantify every problem: missing values, duplicates, and the
unique (messy) values in each categorical column.

In [3]:
missing = raw.isna().sum()
missing = missing[missing > 0].sort_values(ascending=False)
print("Missing values per column:")
print(missing)

print("\nExact duplicate rows:", raw.duplicated().sum())

print("\nUnique raw 'borough' values:", sorted(raw['borough'].astype(str).unique()))
print("\nUnique raw 'cuisine_type' values:", sorted(raw['cuisine_type'].astype(str).unique()))


Missing values per column:
violations_count    58
grade               29
restaurant_name      7
dtype: int64

Exact duplicate rows: 25

Unique raw 'borough' values: [' Bronx', ' Brooklyn', ' Manhattan', 'BRONX', 'BROOKLYN', 'Bklyn', 'Bronx', 'Brooklyn', 'MANHATTAN', 'Manhattan', 'Manhattan ', 'QUEENS', 'Queens', 'Queens ', 'S.I.', 'STATEN ISLAND', 'Staten Island', 'bronx', 'brooklyn', 'manhattan', 'queens', 'staten island']

Unique raw 'cuisine_type' values: ['AMERICAN', 'Amer.', 'American', 'CHINESE', 'Chinese', 'INDIAN', 'ITALIAN', 'Indian', 'Italian', 'Italian ', 'JAPANESE', 'Jap.', 'Japanese', 'MEXICAN', 'Mexican', 'PIZZA', 'Pizza', 'american', 'chinese', 'indian', 'italian', 'japanese', 'mexican', 'pizza']


## 3. Remove exact duplicates

Duplicate inspection rows would double-count restaurants in any aggregation, so they're
dropped first, before any other transformation touches the data.

In [4]:
df = raw.drop_duplicates().copy()
print(f"Dropped {len(raw) - len(df)} duplicate rows -> {len(df)} rows remain")


Dropped 25 duplicate rows -> 600 rows remain


## 4. Standardize text columns

`borough` and `cuisine_type` have the same real values typed in different cases, with
stray whitespace. We normalize by stripping whitespace, upper-casing for comparison,
then mapping back to a clean, title-cased canonical label.

In [5]:
def clean_category(series: pd.Series, canonical: list[str]) -> pd.Series:
    cleaned = series.astype(str).str.strip()
    lookup = {c.upper(): c for c in canonical}
    # also catch common abbreviations seen in this dataset
    manual_fixes = {"BKLYN": "Brooklyn", "S.I.": "Staten Island", "AMER.": "American", "JAP.": "Japanese"}
    def map_value(v):
        key = v.upper()
        if key in manual_fixes:
            return manual_fixes[key]
        return lookup.get(key, v)
    return cleaned.apply(map_value)

boroughs = ["Manhattan", "Brooklyn", "Queens", "Bronx", "Staten Island"]
cuisines = ["Italian", "Chinese", "American", "Mexican", "Indian", "Japanese", "Pizza"]

df["borough"] = clean_category(df["borough"], boroughs)
df["cuisine_type"] = clean_category(df["cuisine_type"], cuisines)

print("Boroughs after cleaning:", sorted(df["borough"].unique()))
print("Cuisines after cleaning:", sorted(df["cuisine_type"].unique()))


Boroughs after cleaning: ['Bronx', 'Brooklyn', 'Manhattan', 'Queens', 'Staten Island']
Cuisines after cleaning: ['American', 'Chinese', 'Indian', 'Italian', 'Japanese', 'Mexican', 'Pizza']


## 5. Parse the four inconsistent date formats

`inspection_date` mixes `YYYY-MM-DD`, `MM/DD/YYYY`, `DD-MM-YYYY`, and `YYYY/MM/DD`.
`pd.to_datetime` with `format="mixed"` infers the format per-row, which is the safest
option when formats aren't labeled.

In [6]:
df["inspection_date"] = pd.to_datetime(df["inspection_date"], format="mixed", dayfirst=False, errors="coerce")
print("Unparseable dates:", df["inspection_date"].isna().sum())
df["inspection_date"].dt.year.value_counts().sort_index()


Unparseable dates: 0


inspection_date
2021    168
2022    132
2023    159
2024    141
Name: count, dtype: int64

## 6. Fix the `violations_count` column (mixed types + missing values)

One stray row stores the word `"two"` instead of a number. We coerce to numeric
(turning unparseable values into `NaN`), then decide how to handle the missing values:
here we impute with the **median**, which is robust to the outlier scores we'll handle
next, and we keep a flag column so the imputation is transparent rather than silently
hiding missing data.

In [7]:
df["violations_count"] = pd.to_numeric(df["violations_count"], errors="coerce")

df["violations_count_was_missing"] = df["violations_count"].isna()
median_violations = df["violations_count"].median()
df["violations_count"] = df["violations_count"].fillna(median_violations)
df["violations_count"] = df["violations_count"].astype(int)

print(f"Imputed {df['violations_count_was_missing'].sum()} missing violation counts with median={median_violations}")


Imputed 58 missing violation counts with median=6.0


## 7. Handle impossible / outlier inspection scores

A valid inspection score is 0-100. We found values of `-5` and `999`, which are data
entry errors, not real extreme scores. Rather than silently imputing, we clip them to
`NaN` and then flag + impute with the column median, which keeps the row (and its other
valid fields) usable while not distorting the score distribution.

In [8]:
invalid_score_mask = (df["score"] < 0) | (df["score"] > 100)
print("Invalid scores found:", invalid_score_mask.sum())

df.loc[invalid_score_mask, "score"] = np.nan
df["score_was_invalid"] = invalid_score_mask
df["score"] = df["score"].fillna(df["score"].median()).astype(int)

df["score"].describe()


Invalid scores found: 24


count    600.000000
mean      30.651667
std       16.915887
min        0.000000
25%       18.000000
50%       31.000000
75%       44.000000
max       59.000000
Name: score, dtype: float64

## 8. Handle missing `restaurant_name` and `grade`

- `restaurant_name` missing -> fill with a clear placeholder (`"Unknown"`) since we can't
  invent a name, and dropping the row would throw away a valid inspection record.
- `grade` missing -> many real inspections genuinely have no grade yet (pending), so we
  fill with the explicit category `"Not Yet Graded"` instead of guessing a letter grade.

In [9]:
df["restaurant_name"] = df["restaurant_name"].fillna("Unknown")
df["grade"] = df["grade"].fillna("Not Yet Graded")

df[["restaurant_name", "grade"]].isna().sum()


restaurant_name    0
grade              0
dtype: int64

## 9. Final validation — confirm the dataset is analysis-ready

In [10]:
assert df.duplicated().sum() == 0
assert df["score"].between(0, 100).all()
assert df["borough"].isin(boroughs).all()
assert df["cuisine_type"].isin(cuisines).all()
assert df["inspection_date"].notna().all()
assert df["violations_count"].dtype == int

print("All validation checks passed.")
df.info()


All validation checks passed.
<class 'pandas.DataFrame'>
RangeIndex: 600 entries, 0 to 599
Data columns (total 10 columns):
 #   Column                        Non-Null Count  Dtype         
---  ------                        --------------  -----         
 0   inspection_id                 600 non-null    int64         
 1   restaurant_name               600 non-null    str           
 2   borough                       600 non-null    str           
 3   cuisine_type                  600 non-null    str           
 4   inspection_date               600 non-null    datetime64[us]
 5   score                         600 non-null    int64         
 6   grade                         600 non-null    str           
 7   violations_count              600 non-null    int64         
 8   violations_count_was_missing  600 non-null    bool          
 9   score_was_invalid             600 non-null    bool          
dtypes: bool(2), datetime64[us](1), int64(3), str(4)
memory usage: 38.8 KB


## 10. Save the cleaned dataset

In [11]:
df.to_csv("data/cleaned_restaurant_inspections.csv", index=False)
print("Saved data/cleaned_restaurant_inspections.csv with", len(df), "rows")
df.head()


Saved data/cleaned_restaurant_inspections.csv with 600 rows


,inspection_id,restaurant_name,borough,cuisine_type,inspection_date,score,grade,violations_count,violations_count_was_missing,score_was_invalid
0,1000,Restaurant 32,Bronx,Mexican,2023-10-20,51,Not Yet Graded,6,False,False
1,1001,Restaurant 130,Bronx,American,2022-11-23,32,A,7,False,False
2,1002,Restaurant 156,Staten Island,Japanese,2023-09-25,11,A,10,False,False
3,1003,Restaurant 130,Bronx,Mexican,2022-02-23,41,A,8,False,False
4,1004,Restaurant 294,Brooklyn,American,2024-10-22,40,A,6,False,False


## 11. Quick before/after summary

A short, reusable summary of what changed, useful for a README or a PR description.

In [12]:
summary = pd.DataFrame({
    "metric": [
        "rows (raw)", "rows (cleaned)", "duplicate rows removed",
        "invalid scores fixed", "missing violations imputed",
        "missing names filled", "missing grades filled",
    ],
    "value": [
        len(raw), len(df), len(raw) - len(df.drop_duplicates()) if False else (raw.duplicated().sum()),
        int(invalid_score_mask.sum()), int(df['violations_count_was_missing'].sum()),
        int(raw['restaurant_name'].isna().sum()), int(raw['grade'].isna().sum()),
    ],
})
summary


,metric,value
0,rows (raw),625
1,rows (cleaned),600
2,duplicate rows removed,25
3,invalid scores fixed,24
4,missing violations imputed,58
5,missing names filled,7
6,missing grades filled,29
